# Lab 09 · A hook that actually blocks
**~20 minutes · costs nothing · Domain 7 Hooks + Security, Domain 3**

The load-bearing idea in three domains: **a prompt instruction is a request, a
hook is a control.** And the detail that catches people: **exit 2 blocks, exit 1
fails open.** A crashing guard script lets the command through.

In [ ]:
import os, json, pathlib, subprocess, sys
root = pathlib.Path("hooklab"); (root/".claude"/"hooks").mkdir(parents=True, exist_ok=True)
print(root.resolve())

## The guard script

In [ ]:
guard = r'''#!/usr/bin/env bash
# Reads the hook payload on stdin. No jq dependency: python3 is always present
# wherever Claude Code runs.
input=$(cat)
command=$(printf '%s' "$input" | python3 -c 'import sys,json; print(json.load(sys.stdin).get("tool_input",{}).get("command",""))' 2>/dev/null)

if printf '%s' "$command" | grep -qiE '(^|[[:space:]])rm[[:space:]]+(-[a-z]*r|--recursive)'; then
  echo "Recursive rm is blocked by project policy." >&2
  exit 2      # BLOCKS. stderr is returned to Claude as the reason.
fi
exit 0        # proceeds
'''
p = root/".claude"/"hooks"/"guard.sh"
p.write_text(guard); p.chmod(0o755)

settings = {"hooks":{"PreToolUse":[{"matcher":"Bash",
  "hooks":[{"type":"command","command":"$CLAUDE_PROJECT_DIR/.claude/hooks/guard.sh"}]}]}}
(root/".claude"/"settings.json").write_text(json.dumps(settings, indent=2))
print(json.dumps(settings, indent=2))

## Test the guard directly

Feed it the same JSON payload Claude Code would, and read the **exit code**.
That is the entire contract.

In [ ]:
def fire(cmd):
    payload = json.dumps({"tool_name":"Bash","tool_input":{"command":cmd}})
    r = subprocess.run([str(p)], input=payload, capture_output=True, text=True)
    verdict = {0:"ALLOW", 2:"BLOCK"}.get(r.returncode, f"HOOK ERROR -> ALLOWED ANYWAY")
    print(f"{cmd:<34} exit={r.returncode}  {verdict}  {r.stderr.strip()}")

for c in ["ls -la", "rm -rf /tmp/x", "rm --recursive build", "rm file.txt", "git status"]:
    fire(c)

## The footgun

Break the script the way a real one breaks: a missing dependency, a typo, an
unset variable. It exits non-zero but **not 2**, so the destructive command
proceeds. Your guard failed **open**.

In [ ]:
broken = root/".claude"/"hooks"/"broken.sh"
broken.write_text("#!/usr/bin/env bash\ncat > /dev/null\nnosuchcommand_typo\n")
broken.chmod(0o755)

r = subprocess.run([str(broken)],
        input=json.dumps({"tool_name":"Bash","tool_input":{"command":"rm -rf /"}}),
        capture_output=True, text=True)
print("exit code:", r.returncode, "->", "BLOCKED" if r.returncode==2 else "ALLOWED")
print()
print("Exit 1 (or 127) is a non-blocking hook error. The tool call still runs.")
print("Guard scripts need their own error handling, or they protect nothing.")

## Try it live

```
cd hooklab && claude
```
Then ask Claude Code to `rm -rf build`. It should refuse and quote your reason.

### Where this is tested
- **Domain 7 Hooks (1.0%)** — the blocking mechanism
- **Domain 7 App Security (3.2%)** — hooks as a structural injection mitigation
- **Domain 1 Agent Construction (5.3%)** — hooks for deterministic actions
- **Domain 3 (3.1%)** — settings.json configuration

---
### Checkpoint
- Which event can block, and which cannot undo?
- What does exit 2 do that exit 1 does not?
- Matchers match against what, exactly?